# 06 - Baseline Model

Assembles a modeling-ready feature table at 15-minute grain
(`station_id`, `datetime`) from the 23 joined bike-count/weather CSVs
(`data/raw/joined/`) plus calendar features, then scores a
**seasonal-naive persistence baseline** ("traffic 24h from now = traffic
right now") on a held-out chronological test set. Baseline numbers here are
the bar any real model must clear.

Uses `muenster_bike_forecast.modeling.model_table` for every pure
transform (channel coalescing, target construction, calendar merge, the
chronological split, and metric computation); this notebook only handles
I/O and orchestration.

Steps:

1. Load all 23 station files, compute `total_count` per row (coalescing
   the duplicate-channel-column issue - see module docstring), report
   data-quality stats.
2. Build the 24h-ahead forecast target via an exact-timestamp lookup.
3. Merge calendar features (public holidays, school holidays, lecture
   periods, hour/day-of-week/month).
4. Save the assembled table to `data/raw/model_table/`.
5. Chronological train/test split with a single global cutoff (last 8
   weeks of data = test).
6. Baseline prediction + MAE/RMSE, overall and per station.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Make `src/` importable regardless of whether this notebook is run from
# `notebooks/` (the normal case) or the project root.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from muenster_bike_forecast.data.calendar import (
    DEFAULT_PUBLIC_HOLIDAY_SUBDIV,
    load_school_holidays,
    public_holidays,
)
from muenster_bike_forecast.modeling.model_table import (
    add_baseline_prediction,
    add_calendar_features,
    add_forecast_target,
    chronological_split,
    compute_baseline_metrics,
    compute_total_count,
    identify_channel_count_columns,
    summarize_baseline_evaluable_rows,
    summarize_target_nulls,
)

JOINED_DIR = PROJECT_ROOT / "data" / "raw" / "joined"
CALENDAR_DIR = PROJECT_ROOT / "data" / "raw" / "calendar"
MODEL_TABLE_DIR = PROJECT_ROOT / "data" / "raw" / "model_table"

TEST_PERIOD = pd.Timedelta(weeks=8)
FORECAST_HORIZON = pd.Timedelta(hours=24)

station_files = sorted(JOINED_DIR.glob("*.csv"))
print(
    f"{len(station_files)} joined station file(s) found in "
    f"{JOINED_DIR.relative_to(PROJECT_ROOT)}"
)

23 joined station file(s) found in data\raw\joined


## 1. Load stations, compute `total_count`, check the duplicate-channel-column issue

For every station file: identify count-columns by leading channel id
(`identify_channel_count_columns`), count how many distinct channel ids
have *more than one* column sharing that id (the source repo renaming a
channel's description mid-history - see module docstring for the concrete
example), then compute `total_count` via `compute_total_count`. Every
station also publishes a "combined" channel (numeric id equal to the
station's own `station_id`) alongside directional sub-channels that
already sum to it (confirmed across all 23 stations via
`combined_channel_matches_directional_sum`) - so `compute_total_count`
returns that combined channel's (coalesced, for the renamed-duplicate
case above) value directly, rather than summing every channel id, which
would double-count.

Only `station_id`, `datetime`, `total_count`, and the `weather_*` feature
columns (dropping the two purely administrative ones,
`weather_station_id` and `weather_timestamp`, which are join bookkeeping,
not features) are kept per station.

In [2]:
def _load_station_slim(path: Path) -> tuple[pd.DataFrame, dict[str, object]]:
    """Loads one joined station CSV and reduces it to modeling columns.

    Args:
        path: Path to one `data/raw/joined/<station_id>.csv` file.

    Returns:
        ``(slim_df, stats)`` where `slim_df` has columns `station_id`,
        `datetime`, `total_count`, and every `weather_*` feature column
        (excluding the administrative `weather_station_id` /
        `weather_timestamp`), and `stats` is a dict with keys
        `station_id`, `n_rows`, `n_channels`, `n_duplicate_channel_ids`
        (channel ids backed by more than one raw column).
    """
    df = pd.read_csv(path)
    df["datetime"] = pd.to_datetime(df["datetime"])

    channel_columns = identify_channel_count_columns(list(df.columns))
    n_duplicate_channel_ids = sum(
        1 for cols in channel_columns.values() if len(cols) > 1
    )

    station_id = df["station_id"].iloc[0]
    total_count = compute_total_count(df, station_id)

    weather_columns = [
        c
        for c in df.columns
        if c.startswith("weather_") and c not in ("weather_station_id", "weather_timestamp")
    ]
    slim = df[["station_id", "datetime", *weather_columns]].copy()
    slim["total_count"] = total_count

    stats = {
        "station_id": station_id,
        "n_rows": len(df),
        "n_channels": len(channel_columns),
        "n_duplicate_channel_ids": n_duplicate_channel_ids,
    }
    return slim, stats


slim_frames: list[pd.DataFrame] = []
station_stats: list[dict[str, object]] = []
for path in station_files:
    slim, stats = _load_station_slim(path)
    slim_frames.append(slim)
    station_stats.append(stats)

full_df = pd.concat(slim_frames, ignore_index=True)
full_df = full_df.sort_values(["station_id", "datetime"]).reset_index(drop=True)

station_stats_df = pd.DataFrame(station_stats)
print(f"Combined table: {len(full_df):,} rows, {full_df.shape[1]} columns, {full_df['station_id'].nunique()} stations")
station_stats_df

Combined table: 2,337,596 rows, 13 columns, 23 stations


,station_id,n_rows,n_channels,n_duplicate_channel_ids
0,100020113,120744,3,1
1,100031297,117463,7,1
2,100031300,221612,3,2
3,100034978,119196,3,0
4,100034980,119296,3,0
5,100034981,119212,3,0
6,100034982,118844,3,0
7,100034983,119178,3,0
8,100035541,119168,3,0
9,100053305,116310,7,7


In [3]:
n_stations_with_dupes = int((station_stats_df["n_duplicate_channel_ids"] > 0).sum())
total_dupe_ids = int(station_stats_df["n_duplicate_channel_ids"].sum())
print(
    f"{n_stations_with_dupes} of {len(station_stats_df)} stations have at least one "
    f"channel id backed by more than one raw column (renamed description "
    f"mid-history); {total_dupe_ids} such channel ids in total across all stations. "
    "compute_total_count() coalesced each of these before selecting the combined "
    "channel's value, so none of them were double-counted."
)

13 of 23 stations have at least one channel id backed by more than one raw column (renamed description mid-history); 53 such channel ids in total across all stations. compute_total_count() coalesced each of these before selecting the combined channel's value, so none of them were double-counted.


## 2. 24h-ahead forecast target (exact-timestamp lookup)

For each `(station_id, datetime)` row, `target_total_count` is the
`total_count` value at the row whose timestamp is *exactly* 24h later for
the *same* station - via a real timestamp lookup (`add_forecast_target`),
not a positional shift, so the data's real 15-minute gaps never silently
pair a row with something that isn't actually 24h later. Rows with no
data at exactly `t + 24h` get a null target.

In [4]:
full_df = add_forecast_target(
    full_df,
    value_col="total_count",
    timestamp_col="datetime",
    station_col="station_id",
    horizon=FORECAST_HORIZON,
    target_col="target_total_count",
)

target_null_summary = summarize_target_nulls(full_df, target_col="target_total_count")
print(target_null_summary)

{'n_rows': 2337596, 'n_null_target': 71765, 'pct_null_target': 3.0700343429745773}


## 3. Calendar features

Merges in `is_public_holiday` (NRW public holidays), `is_school_holiday`
(NRW school-holiday periods, cached at
`data/raw/calendar/school_holidays_nw.csv`), `is_lecture_period` (WWU
semester table), and the cheap timestamp-derived `hour`, `day_of_week`,
`month`.

In [5]:
start_year = int(full_df["datetime"].min().year)
end_year = int(full_df["datetime"].max().year)
print(f"Computing public holidays for {start_year}-{end_year}")

public_holidays_df = public_holidays(
    start_year, end_year, subdiv=DEFAULT_PUBLIC_HOLIDAY_SUBDIV
)
school_holidays_df = load_school_holidays(CALENDAR_DIR / "school_holidays_nw.csv")

full_df = add_calendar_features(full_df, public_holidays_df, school_holidays_df)

full_df[
    ["is_public_holiday", "is_school_holiday", "is_lecture_period"]
].mean().rename("share_of_rows_true")

Computing public holidays for 2020-2026


is_public_holiday    0.032483
is_school_holiday    0.216442
is_lecture_period    0.613599
Name: share_of_rows_true, dtype: float64

## 4. Save the assembled feature table

Written to `data/raw/model_table/` (covered by the existing
`data/raw/*` gitignore pattern - regenerable from this notebook, not
committed).

In [6]:
MODEL_TABLE_DIR.mkdir(parents=True, exist_ok=True)
model_table_path = MODEL_TABLE_DIR / "model_table.csv"
full_df.to_csv(model_table_path, index=False)
print(
    f"Saved {len(full_df):,} rows x {full_df.shape[1]} columns -> "
    f"{model_table_path.relative_to(PROJECT_ROOT)}"
)

Saved 2,337,596 rows x 20 columns -> data\raw\model_table\model_table.csv


## 5. Chronological train/test split

A single global cutoff (`max(datetime)` across *all* stations, minus 8
weeks) applied uniformly across every station - not a per-station
cutoff - so no station's "future" can leak relative to another's.

In [7]:
train_df, test_df, cutoff = chronological_split(
    full_df, timestamp_col="datetime", test_period=TEST_PERIOD
)
print(f"Global max timestamp: {full_df['datetime'].max()}")
print(f"Test-period length:   {TEST_PERIOD}")
print(f"Cutoff (test start):  {cutoff}")
print(f"Train rows: {len(train_df):,}   Test rows: {len(test_df):,}")

test_df["station_id"].value_counts().sort_index().rename("n_test_rows")

Global max timestamp: 2026-07-06 04:45:00
Test-period length:   56 days 00:00:00
Cutoff (test start):  2026-05-11 04:45:00
Train rows: 2,223,556   Test rows: 112,000


station_id
100020113    5225
100031297    5225
100031300    5225
100034978    5209
100034980    5225
100034981    5225
100034982    5225
100034983    5189
100035541    5195
100053305    5183
300037405    5171
300037544    3016
300037920    5189
300037925    3339
300037926    5189
300037928    5171
300037931    4805
300037932    3146
300037933    5189
300037936    5192
300038855    4053
300039328    5225
300039331    5189
Name: n_test_rows, dtype: int64

## 6. Seasonal-naive baseline: prediction + evaluability

The baseline needs no fitting: predicted `total_count` 24h ahead = the
row's own current `total_count`. Evaluated only on test rows where both
the current value and the true 24h-ahead target are non-null.

In [8]:
test_df = add_baseline_prediction(
    test_df, current_col="total_count", prediction_col="baseline_prediction"
)
evaluable_summary = summarize_baseline_evaluable_rows(
    test_df, prediction_col="baseline_prediction", target_col="target_total_count"
)
print(evaluable_summary)

{'n_rows': 112000, 'n_evaluable': 106043, 'n_excluded': 5957, 'pct_excluded': 5.31875}


## 7. Baseline metrics: MAE / RMSE, overall and per station

Computed on the test set only, so a future model can be compared on
exactly the same held-out period.

In [9]:
overall_metrics = compute_baseline_metrics(
    test_df, prediction_col="baseline_prediction", target_col="target_total_count"
)
per_station_metrics = compute_baseline_metrics(
    test_df,
    prediction_col="baseline_prediction",
    target_col="target_total_count",
    group_col="station_id",
)

metrics_table = pd.concat([overall_metrics, per_station_metrics], ignore_index=True)
metrics_table

,group,mae,rmse,n_rows
0,overall,19.670172,38.626091,106043
1,100020113,17.572031,26.008909,4977
2,100031297,57.761704,91.574395,4977
3,100031300,27.053044,40.607955,4977
4,100034978,11.601011,17.968307,4945
5,100034980,31.520796,48.203638,4977
6,100034981,12.743018,19.950884,4977
7,100034982,32.945550,53.222255,4977
8,100034983,24.600569,37.942993,4917
9,100035541,38.116095,59.337555,4927


## Summary

- Assembled `full_df`: one row per `(station_id, datetime)` at 15-minute
  resolution across 23 stations, with `total_count`, the 24h-ahead
  `target_total_count`, calendar features, and the already-joined
  `weather_*` columns. Saved to `data/raw/model_table/model_table.csv`.
- **`total_count` is now each station's combined channel value directly**
  (2026-08-21 fix): every station publishes a "combined" count channel
  (numeric id equal to the station's own `station_id`) alongside two or
  more directional sub-channels that already sum to it - confirmed across
  all 23 stations, zero exceptions, via
  `combined_channel_matches_directional_sum`. `compute_total_count`
  previously summed every channel id it found, double-counting every
  station's traffic by including that redundant combined channel on top
  of the directional breakdown. **This notebook has not been re-run
  since that fix landed** - the numbers printed above (including the
  duplicate-channel-id stats and baseline metrics) still reflect the old,
  double-counted `total_count`; a fresh `Restart & Run All` is needed
  before `data/raw/model_table/model_table.csv` and every number in this
  notebook reflect the corrected value.
- Duplicate-channel-column issue (a channel id backed by two differently
  -named columns due to a mid-history description rename/typo-fix in the
  source repo) is a separate, still-real issue - see the per-station
  counts above - and is still handled by coalescing same-id columns, so
  no channel's value is ever lost or double-counted for that reason
  either.
- `target_total_count` is null for the fraction of rows reported above
  (mostly the last 24h of each station's coverage, where "tomorrow" data
  doesn't exist yet).
- Chronological split: a single global cutoff (`max(datetime)` across all
  stations minus 8 weeks), printed above, applied uniformly to every
  station.
- Baseline (seasonal-naive persistence) MAE/RMSE, overall and per
  station, printed above - this is the number any real 24h-ahead model
  must beat. **Once this notebook is re-run, these will not be directly
  comparable to previously-published baseline numbers** (e.g. CLAUDE.md's
  historical 39.34/77.25), since the target scale itself changes with the
  double-counting fix above.